## Vision Language Models

A **Vision Language Model (VLM)** is an AI system built by combining a large language model (LLM) with a vision encoder, giving the LLM the ability to “see”.

As such, the task it dubbed [Image-Text-to-Text](https://huggingface.co/models?pipeline_tag=image-text-to-text&sort=trending) on HuggingFace (shows `26,049` models). For examples see [Overview of Open-source Vision Language Models | HuggingFace Blog](https://huggingface.co/blog/vlms#overview-of-open-source-vision-language-models).


## VLM Input-output

Specifically:

- Input: Image  & Text
- Output: Text

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/vlm/visual.jpg)

**Note:** Some models also output bounding boxes (object localization) or pixel masks (image segmentation).


## VLM Key Capabilities

VLMs bridge perception and language. Models see pixels then speak:

1.  **Multimodal Learning**: this multiplies information (a picture is worth a thousand words).
2.  **Zero-shot**: you can ask anything about an image.
3.  **Everything can be an image**: a pdf, a web page, a word-document, a diagram.  This broadens functional scope.

## Task 1: Information Extraction with a VLM

Your task is to run and read the notebook to apply VLMs to extract **structured output** from visual documents.

This example can be your starting point [VLM pipeline with GraniteDocling](https://docling-project.github.io/docling/examples/minimal_vlm_pipeline/) as well as this [Vision models Usage Page](https://docling-project.github.io/docling/usage/vision_models/).

Note: A GPU is needed to run a VLM (locally on Colab or Remotely via [OpenRouter.ai](https://openrouter.ai/models?fmt=cards&input_modalities=image&max_price=0&output_modalities=text) or [HuggingFace](https://huggingface.co/blog/vlms#finding-the-right-vision-language-model)).

[Colab: Information Extraction](https://colab.research.google.com/github/docling-project/docling/blob/main/docs/examples/extraction.ipynb#scrollTo=932f12cd).

Input Examples:

- `reciept.png`
- `resume.pdf`
- `form.docx`

## Task 2: Arabic Document Processing

[QARI-OCR](https://huggingface.co/NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct) is a **Vision-language Model (VLM)** fine-tuned from `Qwen2-VL-2B-Instruct` to process Arabic documents.

Key Features:

- 📐 **Layout-Aware Recognition**: Preserves document structure with HTML/Markdown tags
- 🔤 **Full Diacritics Support**: Accurate recognition of tashkeel (Arabic diacritical marks)
- 📝 **Multi-Font Handling**: Trained on 12 diverse Arabic fonts (14px-100px)
- 🎯 **Structure-First Design**: Optimized for documents with headers, body text, and complex layouts
- ⚡ **Efficient Training**: Only 11 hours on single GPU with 10k samples
- 🖼️ **Robust Performance**: Handles low-resolution and degraded images

Your task is to use this model to process a folder of Arabic PDF documents, and extract information from them.

Input example:

- `reciept.png`
- `some_email.jpg`
- `my_resume.pdf`
- `data/HR/applications/ar/` (folder)



---



## Task 1

In [2]:
%pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 92.2 MB/s eta 0:00:00
   ━

In [3]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmConvertOptions,
    VlmPipelineOptions,
)
from docling.datamodel.vlm_engine_options import (
    MlxVlmEngineOptions,
    VlmEngineType,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline

In [5]:
# Convert a public arXiv PDF; replace with a local path if preferred.
source = "/content/image_text_example.jpg"

In [6]:
###### EXAMPLE 1: USING DEFAULT SETTINGS (SIMPLEST)
# - No configuration needed
# - Uses default VLM model (GraniteDocling)
# - Auto-selects the best runtime for your platform

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,
        ),
    }
)

doc = converter.convert(source=source).document

print(doc.export_to_markdown())

[INFO] 2026-04-30 03:06:37,300 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:06:37,310 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:06:37,321 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:06:38,170 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-30 03:06:38,420 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:06:38,422 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:06:38,991 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:06:38,993 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:06:38,994 [RapidOCR] download_file.py:68: Init

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

<!-- image -->

Languagetestersare sometimes asked to saywhatis'thebesttest'or'the best testing technique'Such questionsreveal a misunderstanding ofwhat is involved in the practice of language testing. A test that proves ideal for onepurpose may be quite useless for another; a technique that may work verywell in one situation canbe entirely inappropriate in another.What teaching institutions. Equally, two teaching institutions may require very different tests, depending on the objectives of their courses, the purpose of the tests,and the resources available.Each testing situation is unique and setsaparticular testingproblem.And sothefirst stepmustbe tostate this testingproblem as clearly aspossible.Whatever test or testingsystem we thencreateshouldbeonethat:

- consistentlyprovides accurate measures ofprecisely the abilities'in whichwe areinterested;
- hasapositiveinfluence onteaching(inthose caseswherethetestis likely to influence teaching);
- is economical in terms of time and money.

In [7]:
###### EXAMPLE 2: USING PRESETS (RECOMMENDED)
# - Uses the "granite_docling" preset explicitly
# - Same as default but more explicit and configurable
# - Auto-selects the best runtime for your platform (Transformers by default)

vlm_options = VlmConvertOptions.from_preset("granite_docling")

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,
            pipeline_options=VlmPipelineOptions(vlm_options=vlm_options),
        ),
    }
)

doc = converter.convert(source=source).document

print(doc.export_to_markdown())


[INFO] 2026-04-30 03:09:08,482 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:09:08,483 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:09:08,529 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:09:08,530 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:09:08,955 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:09:08,957 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:09:08,963 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-30 03:09:08,967 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

<!-- image -->

Languagetestersare sometimes asked to saywhatis'thebesttest'or'the best testing technique'Such questionsreveal a misunderstanding ofwhat is involved in the practice of language testing. A test that proves ideal for onepurpose may be quite useless for another; a technique that may work verywell in one situation canbe entirely inappropriate in another.What teaching institutions. Equally, two teaching institutions may require very different tests, depending on the objectives of their courses, the purpose of the tests,and the resources available.Each testing situation is unique and setsaparticular testingproblem.And sothefirst stepmustbe tostate this testingproblem as clearly aspossible.Whatever test or testingsystem we thencreateshouldbeonethat:

- consistentlyprovides accurate measures ofprecisely the abilities'in whichwe areinterested;
- hasapositiveinfluence onteaching(inthose caseswherethetestis likely to influence teaching);
- is economical in terms of time and money.

In [8]:
###### EXAMPLE 3: USING PRESETS WITH RUNTIME OVERRIDE (ADVANCED)
# Demonstrates using the same preset but overriding the runtime to use MLX
# on macOS with MPS acceleration. The preset automatically uses the MLX-specific
# model variant when available.

vlm_options = VlmConvertOptions.from_preset(
    "granite_docling",
    engine_options=MlxVlmEngineOptions(),
)

# The preset automatically selects the MLX-optimized model variant
print(f"Using model: {vlm_options.model_spec.get_repo_id(VlmEngineType.MLX)}")

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,
            pipeline_options=VlmPipelineOptions(vlm_options=vlm_options),
        ),
    }
)

doc = converter.convert(source=source).document

print(doc.export_to_markdown())

[INFO] 2026-04-30 03:09:51,112 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:09:51,114 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:09:51,167 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-30 03:09:51,168 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth


Using model: ibm-granite/granite-docling-258M-mlx


[INFO] 2026-04-30 03:09:51,550 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:09:51,552 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:09:51,558 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-30 03:09:51,560 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-30 03:09:51,726 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-30 03:09:51,727 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-04-30 03:09:51,924 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-04-30 03:09:51,927 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

<!-- image -->

Languagetestersare sometimes asked to saywhatis'thebesttest'or'the best testing technique'Such questionsreveal a misunderstanding ofwhat is involved in the practice of language testing. A test that proves ideal for onepurpose may be quite useless for another; a technique that may work verywell in one situation canbe entirely inappropriate in another.What teaching institutions. Equally, two teaching institutions may require very different tests, depending on the objectives of their courses, the purpose of the tests,and the resources available.Each testing situation is unique and setsaparticular testingproblem.And sothefirst stepmustbe tostate this testingproblem as clearly aspossible.Whatever test or testingsystem we thencreateshouldbeonethat:

- consistentlyprovides accurate measures ofprecisely the abilities'in whichwe areinterested;
- hasapositiveinfluence onteaching(inthose caseswherethetestis likely to influence teaching);
- is economical in terms of time and money.

---

## Task 2:

In [11]:
!pip install transformers qwen_vl_utils accelerate>=0.26.0 PEFT -U
!pip install -U bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [13]:
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
import os
from qwen_vl_utils import process_vision_info
import requests

In [15]:
path = "/content/text_to_test_arabic.PNG"
image = Image.open(path)

In [16]:
model_name = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
                model_name,
                torch_dtype="auto",
                device_map="auto"
            )
processor = AutoProcessor.from_pretrained(model_name)
max_tokens = 2000

prompt = "Below is the image of one page of a document, as well as some raw textual content that was previously extracted for it. Just return the plain text representation of this document as if you were reading it naturally. Do not hallucinate."
image.save("image.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "file://image.png"},
            {"type": "text", "text": prompt},
        ],
    }
]
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")
generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
os.remove("image.png")
print(output_text)

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

<h3>أُسْرَتَمِي</h3><br><p>كانت أمي قدوةً، قدوتنا . عاشت، مدة عشرين عاماً، مستقيمة، من دون أن تَشتكي أبداً، مع أنها كانت تعاني أكثر منّا، إن كان <b>ذلك</b> ممكناً، لم تكن تطيق أنْ تُفصَلَ عن أولادها ، وتبكي في <i>السرِّ</i> لأننا <i>جَوْعى،</i> ولأننا <i>ينقصنا</i> كلُّ شيء، ولأن <i>ذلك</i> السجن يسلب <i>مِنّا</i> صِبانا وشُبابنا .</p><br><h3>ونفثْتْ <b>فينا،</b> إلى جانب <i>الكرامة،</i> الشجاعةَ، كانت هي الكاميكاز، الهروب كان فكرتها . كانت تُدركُ المخاطر المحتملَة، وتعرف أنها <i>يمكن</i> أنْ تفقدنا في تلك المغامرة، غير أن قناعتها ظلّت ثابتة، لا تهتز .</h3><br><h3>فهمتُ، في أثناء تلك السنوات الفظيعة التي قضيناها نتواصل من دون أن يرى بعضنا <i>بعضاً،</i> أهميةَ الصوت . كنت <i>أسمعُ</i> صوتها، من خلف <i>الجدار،</i> فأُدركُ، ممّا يطرأ عليه من تغيّرات <i>طفيفة</i> في رنّته، ما <i>يعجز</i> عن <i>إخباري</i> به خطابٌ طويل، عن حالتها <i>الراهنة</i> . وكانت <i>تصنع</i> الأمر نفسه معي . كانت تتفرّجُ على <b>حياتي،</b> من دون أن تستطيع لها تغييراً أو تحويراً .</h3><br><p>كانت علاقتنا دائماً قوية جد

In [25]:
from bs4 import BeautifulSoup

In [28]:

soup = BeautifulSoup(output_text, 'html.parser')

#Extracting data
print(soup.text)

أُسْرَتَمِيكانت أمي قدوةً، قدوتنا . عاشت، مدة عشرين عاماً، مستقيمة، من دون أن تَشتكي أبداً، مع أنها كانت تعاني أكثر منّا، إن كان ذلك ممكناً، لم تكن تطيق أنْ تُفصَلَ عن أولادها ، وتبكي في السرِّ لأننا جَوْعى، ولأننا ينقصنا كلُّ شيء، ولأن ذلك السجن يسلب مِنّا صِبانا وشُبابنا .ونفثْتْ فينا، إلى جانب الكرامة، الشجاعةَ، كانت هي الكاميكاز، الهروب كان فكرتها . كانت تُدركُ المخاطر المحتملَة، وتعرف أنها يمكن أنْ تفقدنا في تلك المغامرة، غير أن قناعتها ظلّت ثابتة، لا تهتز .فهمتُ، في أثناء تلك السنوات الفظيعة التي قضيناها نتواصل من دون أن يرى بعضنا بعضاً، أهميةَ الصوت . كنت أسمعُ صوتها، من خلف الجدار، فأُدركُ، ممّا يطرأ عليه من تغيّرات طفيفة في رنّته، ما يعجز عن إخباري به خطابٌ طويل، عن حالتها الراهنة . وكانت تصنع الأمر نفسه معي . كانت تتفرّجُ على حياتي، من دون أن تستطيع لها تغييراً أو تحويراً .كانت علاقتنا دائماً قوية جدّاً: كنا نتواطأُ حتى في الألم . منذ ولادتي لم تربطني بها سوى علاقات فاجعة، وانفعالية . كانت تُعذّبُها فكرةُ أنني لن يكون لي أبناء . كان ذلك، في نظرها، جزءاً من اللعنة التي رافقتني